In [38]:
import json
from pathlib import Path

from mobile_motion_planning.ik_offline.geometry import Plane
from mobile_motion_planning.partial_trajectory import calculate_partial_trajectory

from mobile_motion_planning.ik_offline.ik_offline import (
    ik_no_tool_with_base_change,
    find_matching_ik_solution,
)

In [ ]:
# replan for 10 nodes, then print out some details about the resulting trajectory and IK solutions
target_planes_json = Path(
    "/home/robot/robot_ws/src/print_while_driving_packages/mobile_motion_planning/data/example_data/260311_Segment_4/260311_150455_flange_frames.json"
 )
base_planes_json = Path(
    "/home/robot/robot_ws/src/print_while_driving_packages/mobile_motion_planning/data/example_data/260311_Segment_4/260311_150455_base_frames.json"
 )

def load_planes_from_json(json_path: Path):
    with json_path.open("r", encoding="utf-8") as handle:
        payload = json.load(handle)

    rows = payload.get("planes") if isinstance(payload, dict) else payload
    if not isinstance(rows, list):
        raise ValueError(f"Expected a list of planes in {json_path}")

    planes = []
    for idx, row in enumerate(rows):
        if isinstance(row, dict):
            point = row.get("origin") or row.get("point")
            xaxis = row.get("x_axis") or row.get("xaxis")
            yaxis = row.get("y_axis") or row.get("yaxis")
        elif isinstance(row, list) and len(row) == 3:
            point, xaxis, yaxis = row
        else:
            raise ValueError(f"Invalid plane at index {idx} in {json_path}")

        if not point or not xaxis or not yaxis:
            raise ValueError(
                f"Plane at index {idx} missing origin/x_axis/y_axis in {json_path}"
            )
        planes.append(Plane(tuple(point), tuple(xaxis), tuple(yaxis)))
    return planes

target_planes = load_planes_from_json(target_planes_json)
base_planes = load_planes_from_json(base_planes_json)

print(f"Loaded target planes: {len(target_planes)}")
print(f"Loaded base planes: {len(base_planes)}")
print("First target plane origin:", target_planes[0].origin if target_planes else None)
print("First base plane origin:", base_planes[0].origin if base_planes else None)

partial_trajectory = calculate_partial_trajectory(
    list_of_targets=target_planes,
    base_planes=base_planes,
    number_of_nodes_to_calculate=10,
 )
ik_counts = [len(s) for s in partial_trajectory["ik_solutions_per_node"]]
print("IK solutions in first 10 nodes:", ik_counts)
print("Num configs:", len(partial_trajectory.get("configurations", [])))
print("Path length:", partial_trajectory.get("path_length"))
print("Error:", partial_trajectory.get("error"))

ik_solutions = ik_no_tool_with_base_change(target_planes[1], base_planes[1])
print(f"IK solutions for target plane 1 and base plane 1: {len(ik_solutions)}")

: 

In [ ]:
# debugging
# import inspect
# import mobile_motion_planning.partial_trajectory as pt

# sig = inspect.signature(calculate_partial_trajectory)
# print("Imported from:", pt.__file__)
# print("Signature:", sig)

# bound = sig.bind_partial(target_planes, base_planes)
# print("Param order:", list(sig.parameters.keys()))
# print("Positional arg #1 type:", type(bound.args[0]).__name__)
# print("Positional arg #2 type:", type(bound.args[1]).__name__)
# print("Interpretation:")
# for i, name in enumerate(list(sig.parameters.keys())[:2]):
#     print(f"  arg{i+1} -> {name}")

Imported from: /home/robot/robot_ws/install/mobile_motion_planning/lib/python3.12/site-packages/mobile_motion_planning/partial_trajectory.py
Signature: (list_of_targets, current_pose=None, number_of_nodes_to_calculate=10, base_planes=None)
Param order: ['list_of_targets', 'current_pose', 'number_of_nodes_to_calculate', 'base_planes']
Positional arg #1 type: list
Positional arg #2 type: list
Interpretation:
  arg1 -> list_of_targets
  arg2 -> current_pose


In [ ]:
# Check data paths and log file existence
import os
import slab_net_zero.utilities.utils as snz_utils

snz_data_path = snz_utils.get_data_path()
snz_log_dir = os.path.join(snz_data_path, "auto_generated")
snz_log_file = os.path.join(snz_log_dir, "compute_times.log")

print("slab_net_zero data_path:", snz_data_path)
print("data_path exists:", os.path.isdir(snz_data_path))
print("auto_generated dir exists:", os.path.isdir(snz_log_dir))
print("compute_times.log exists:", os.path.isfile(snz_log_file))

: 

In [36]:
from pathlib import Path

log_file = Path(snz_log_file)
log_file.parent.mkdir(parents=True, exist_ok=True)
log_file.touch(exist_ok=True)

print("Created/verified:", log_file)

Created/verified: /home/robot/robot_ws/src/.venv/lib/python3.12/data/auto_generated/compute_times.log


In [ ]:
# verify odom distance tracking
import math
import threading
import rclpy
from rclpy.node import Node
from nav_msgs.msg import Odometry

ODOM_TOPIC = '/robot/robotnik_base_control/odom'
DURATION_SEC = 120  # how long to listen

class OdomDistanceTracker(Node):
    def __init__(self):
        super().__init__('odom_distance_tracker')
        self._prev = None
        self.distance = 0.0
        self.samples = 0
        self.create_subscription(Odometry, ODOM_TOPIC, self._cb, 10)

    def _cb(self, msg):
        pos = msg.pose.pose.position
        cur = (pos.x, pos.y, pos.z)
        if self._prev is not None:
            self.distance += math.dist(self._prev, cur)
        self._prev = cur
        self.samples += 1

rclpy.init()
node = OdomDistanceTracker()

def spin():
    rclpy.spin(node)

t = threading.Thread(target=spin, daemon=True)
t.start()

import time
time.sleep(DURATION_SEC)

print(f"Samples received : {node.samples}")
print(f"Distance traveled: {node.distance:.4f} m")

node.destroy_node()
rclpy.shutdown()


Samples received : 5943
Distance traveled: 0.6858 m
